In [3]:
import json
import os

# =============================
# PATHS
# =============================

BASE_PATH = r"C:\Users\User\Downloads\archive"

ANNOTATION_FILE = os.path.join(BASE_PATH, "annotations_resized.json")

DATASET_DIR = os.path.join(BASE_PATH, "dataset")

LABELS_DIR = os.path.join(BASE_PATH, "labels")

os.makedirs(LABELS_DIR, exist_ok=True)

# create label folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(LABELS_DIR, split), exist_ok=True)

# =============================
# LOAD COCO DATA
# =============================

print("Loading annotations...")

with open(ANNOTATION_FILE) as f:
    data = json.load(f)

images = {img["id"]: img for img in data["images"]}
annotations = data["annotations"]

# =============================
# GROUP ANNOTATIONS BY IMAGE
# =============================

ann_by_image = {}

for ann in annotations:
    ann_by_image.setdefault(ann["image_id"], []).append(ann)

print("Annotations grouped.")

# =============================
# PROCESS EACH SPLIT
# =============================

for split in ["train", "val", "test"]:

    split_image_dir = os.path.join(DATASET_DIR, split)
    split_label_dir = os.path.join(LABELS_DIR, split)

    print(f"\nProcessing {split} set...")

    for image_file in os.listdir(split_image_dir):

        image_id = None

        # find image id
        for img in images.values():
            if img["file_name"].replace("/", "_") == image_file:
                image_id = img["id"]
                break

        if image_id is None:
            continue

        img_info = images[image_id]

        width = img_info["width"]
        height = img_info["height"]

        label_file = image_file.replace(".jpg", ".txt").replace(".JPG", ".txt")

        label_path = os.path.join(split_label_dir, label_file)

        with open(label_path, "w") as f:

            anns = ann_by_image.get(image_id, [])

            for ann in anns:

                x, y, w, h = ann["bbox"]

                # convert COCO → YOLO
                cx = (x + w / 2) / width
                cy = (y + h / 2) / height
                nw = w / width
                nh = h / height

                class_id = ann["category_id"] - 1

                f.write(f"{class_id} {cx} {cy} {nw} {nh}\n")

print("\nYOLO label generation complete.")

Loading annotations...
Annotations grouped.

Processing train set...

Processing val set...

Processing test set...

YOLO label generation complete.
